# RQ3: Does YouTube comment sentiment reflect voter preferences or amplify extreme views?

In [ ]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["youtube_db"]
collection = db["channels"]
pipeline = [
    # Flatten videos array
    {"$unwind": "$videos"},

    # Flatten comments array
    {"$unwind": "$videos.comments"},

    # Group data
    {
        "$group": {
            "_id": {
                "channel_name": "$channel_name",
                "lean": "$videos.lean",
                "year": "$videos.year"
            },
            "avg_sentiment": {
                "$avg": "$videos.comments.sentiment_score"
            },
            "comment_count": {"$sum": 1}
        }
    },
    
    {
        "$project": {
            "_id": 0,
            "channel_name": "$_id.channel_name",
            "lean": "$_id.lean",
            "year": "$_id.year",
            "avg_sentiment": {"$round": ["$avg_sentiment", 3]},
            "comment_count": 1
        }
    },

    # Sort
    {
        "$sort": {
            "year": 1,
            "lean": 1,
            "channel_name": 1
        }
    }
]

# Run aggregation
results = list(collection.aggregate(pipeline))

# Convert to dataframe
rq3_df = pd.DataFrame(results)

print(rq3_df.head())

   comment_count     channel_name          lean  year  avg_sentiment
0            108      Ben Shapiro  conservative  2018          0.229
1              5  The Young Turks       liberal  2018         -0.025
2            108      Ben Shapiro  conservative  2019          0.240
3              2  The Young Turks       liberal  2019         -0.028
4            108      Ben Shapiro  conservative  2020          0.043
